### Gathering and Plotting Montreal Neighbourhood Housing Census Data from 1971-2021

In [2]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [3]:
montreal_tract_2021 = pd.read_csv("Raw data/montreal_tract_data_2021_CHASS.csv")
montreal_tract_2016 = pd.read_csv("Raw data/montreal_tract_data_2016_CHASS.csv")

montreal_geo_2021 = gpd.read_file("Tracts/2021_tract_Geo/lct_000b21a_e.shp")
montreal_geo_2016 = gpd.read_file("Tracts/2016_tract_Geo/lct_000b16a_e.shp")

def tract_formatting(df: pd.DataFrame):
    df.rename(columns={"COL0": "CTUID",
        "COL1": "Province code",
        "COL2": "Province name",
        "COL3": "CMA code",
        "COL4": "CMA name",
        "COL5": "Census Tract name",
        "COL6": "Total private dwellings",
        "COL7": "Private households by tenure",
        "COL8": "Average number of rooms per dwelling",
        "COL9": "Private households by housing suitability",
        "COL10": "Occupied private dwellings by dwelling condition",
        "COL11": "Private households by number of household maintainers",
        "COL12": "Private households by age of primary household maintainers",
        "COL13": "Spending less than 30% of income on shelter costs",
        "COL14": "Spending 30% or more of income on shelter costs",
        "COL15": "Percentage of owner households with a mortgage",
        "COL16": "Median monthly shelter costs for owned dwellings ($)",
        "COL17": "Average monthly shelter costs for owned dwellings ($)",
        "COL18": "Median value of dwellings ($)",
        "COL19": "Average value of dwellings ($)",
        "COL20": "Percentage of tenant households in subsidized housing"
    }, inplace=True)

    df["CTUID"] = df["CTUID"].apply(lambda x: float(x))

def geo_tract_formatting(gdf: gpd.GeoDataFrame):
    gdf["CTUID"] = gdf["CTUID"].apply(lambda x: float(x))
    return gdf[(gdf["CTUID"] >= 4620000) & (gdf["CTUID"] < 4630000)].to_crs("EPSG:4326")


tract_formatting(montreal_tract_2021)
tract_formatting(montreal_tract_2016)

merged_2021 = geo_tract_formatting(montreal_geo_2021).merge(montreal_tract_2021, on="CTUID", how="left")
merged_2016 = geo_tract_formatting(montreal_geo_2016).merge(montreal_tract_2016, on="CTUID", how="left")

In [92]:
def house_price_formatting(gdf: gpd.GeoDataFrame, year: int):
    new = gdf[["CTUID", "Average value of dwellings ($)", "geometry"]]
    new.rename(columns={"Average value of dwellings ($)": year}, inplace=True)
    return new

house_price_data_2021 = house_price_formatting(merged_2021, 2021)
house_price_data_2016 = house_price_formatting(merged_2016, 2016)

housing_data_final = house_price_data_2021.merge(house_price_data_2016, on=["CTUID", "geometry"], how="left")
housing_data_final = housing_data_final.melt(id_vars=["CTUID", "geometry"], var_name="Year", value_name="Average value of dwellings ($)")
housing_data_final = housing_data_final.set_index("CTUID")
housing_data_final


C:\Users\sando\AppData\Local\Temp\ipykernel_2844\2173461538.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new.rename(columns={"Average value of dwellings ($)": year}, inplace=True)
C:\Users\sando\AppData\Local\Temp\ipykernel_2844\2173461538.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new.rename(columns={"Average value of dwellings ($)": year}, inplace=True)


,geometry,Year,Average value of dwellings ($)
CTUID,,,
4620725.07,"POLYGON ((-73.93606 45.58568, -73.93672 45.584...",2021,353600.0
4620652.04,"POLYGON ((-73.80088 45.53048, -73.80078 45.530...",2021,456400.0
4620887.05,"POLYGON ((-73.43923 45.57119, -73.43982 45.571...",2021,482400.0
4620887.06,"POLYGON ((-73.43049 45.56939, -73.43133 45.569...",2021,543500.0
4620887.07,"POLYGON ((-73.41092 45.57275, -73.4109 45.5726...",2021,NaN
...,...,...,...
4620805.02,"POLYGON ((-73.72078 45.35059, -73.72123 45.349...",2016,NaN
4620685.09,"POLYGON ((-73.61341 45.74301, -73.61335 45.742...",2016,NaN
4620685.10,"POLYGON ((-73.61375 45.73401, -73.61475 45.732...",2016,NaN


In [95]:
fig = px.choropleth_map(housing_data_final, geojson=housing_data_final.geometry.to_json(), color="Average value of dwellings ($)",
                        range_color=(0, 2000000),
                        locations=housing_data_final.index,
                        center={"lat": 45.5517, "lon": -73.7073},
                        map_style="carto-positron", zoom=8)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

ValueError: Series index must be unique for orient='index'